# SESSION 18: Interview Questions + Final Revision

**Course:** Working with Databases & SQL Basics  
**Topic:** Core SQL Interview Mastery, Query Optimization, and Performance Tuning  
**Database:** `interview_revision.db`  
**Execution Environment:** Python 3.12, SQLite 3.49, Pandas 3.1

---

## 📌 Session Overview & Objectives
This final capstone revision synthesizes the most common and critical SQL technical interview problems encountered in Data Analyst, Data Engineer, and Data Scientist hiring loops across tech companies like Flipkart, Amazon, Google, and Zomato.

### Core Topics Tested:
1. **Aggregations & Filter Post-Aggregation (`HAVING`)**: Filtering grouped metrics where row-level `WHERE` cannot be applied.
2. **Relational Join Semantics**: Aggregating one-to-many relationships while preserving un-ordered parent entities (`LEFT JOIN` + `COALESCE`).
3. **Subqueries**: Deploying uncorrelated scalar subqueries in `WHERE` filtering against calculated statistical thresholds.
4. **Window Functions**: Calculating rolling and cumulative sums partitioned by user without collapsing rows.
5. **Database Indexing & Query Tuning**: Designing optimal composite B-Tree indexes and structuring queries for high-throughput e-commerce catalogs.

---

## Task 1: Songs Uploaded per Artist (`GROUP BY` & `HAVING`)
> **Task 1:** Write an SQL query to display the total number of songs uploaded by each artist from a table `songs` (columns: `song_id`, `artist_name`, `title`) and show only those artists who have uploaded more than 3 songs.

### Interview Key Concept: `WHERE` vs `HAVING`
- `WHERE` filters individual rows **before** any aggregation takes place.
- `HAVING` filters aggregated groups **after** `GROUP BY` and aggregate functions (`COUNT`, `SUM`, `AVG`) have run.
- Attempting `WHERE COUNT(song_id) > 3` causes a syntax error because aggregate results do not exist during the `WHERE` evaluation phase.

In [1]:
import sqlite3
import pandas as pd

# Connect to database
conn = sqlite3.connect("interview_revision.db")
cursor = conn.cursor()

# 1. Create songs table
cursor.execute("DROP TABLE IF EXISTS songs;")
cursor.execute("""
CREATE TABLE songs (
    song_id INTEGER PRIMARY KEY,
    artist_name TEXT NOT NULL,
    title TEXT NOT NULL
);
""")
conn.commit()

# 2. Insert sample song uploads
songs_data = [
    (1,  "Arijit Singh",   "Tum Hi Ho"),
    (2,  "Arijit Singh",   "Kesariya"),
    (3,  "Arijit Singh",   "Channa Mereya"),
    (4,  "Arijit Singh",   "Raabta"),
    (5,  "Arijit Singh",   "Kalank"),
    (6,  "Taylor Swift",   "Cruel Summer"),
    (7,  "Taylor Swift",   "Anti-Hero"),
    (8,  "Taylor Swift",   "Blank Space"),
    (9,  "Taylor Swift",   "Lover"),
    (10, "Ed Sheeran",     "Shape of You"),
    (11, "Ed Sheeran",     "Perfect"),
    (12, "Ed Sheeran",     "Bad Habits"),
    (13, "The Weeknd",     "Blinding Lights"),
    (14, "The Weeknd",     "Starboy"),
    (15, "Diljit Dosanjh", "Lover")
]
cursor.executemany("INSERT INTO songs (song_id, artist_name, title) VALUES (?, ?, ?);", songs_data)
conn.commit()

# 3. Query artists with > 3 songs
query_task1 = """
SELECT 
    artist_name,
    COUNT(song_id) AS total_songs
FROM songs
GROUP BY artist_name
HAVING COUNT(song_id) > 3
ORDER BY total_songs DESC, artist_name ASC;
"""
df_task1 = pd.read_sql_query(query_task1, conn)
print("--- Artists with More Than 3 Songs ---")
display(df_task1)


--- Artists with More Than 3 Songs ---


,artist_name,total_songs
0,Arijit Singh,5
1,Taylor Swift,4


## Task 2: Total Order Amount per Username (Relational JOIN)
> **Task 2:** Given two tables, `orders` (`order_id`, `user_id`, `amount`) and `users` (`user_id`, `username`), write a SQL JOIN query to display each `username` along with their total order amount.

### Interview Key Concept: Handling Users with Zero Orders
- If an interviewer asks for **each username**, using an `INNER JOIN` accidentally drops users who have never placed an order.
- In production, a `LEFT JOIN` combined with `COALESCE(SUM(o.amount), 0.00)` ensures all registered users appear on the report, substituting `NULL` with `0.00`.

In [2]:
# 1. Create users and orders tables
cursor.execute("DROP TABLE IF EXISTS orders;")
cursor.execute("DROP TABLE IF EXISTS users;")
cursor.execute("""
CREATE TABLE users (
    user_id INTEGER PRIMARY KEY,
    username TEXT NOT NULL
);
""")
cursor.execute("""
CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    user_id INTEGER NOT NULL,
    amount DECIMAL(10, 2) NOT NULL,
    FOREIGN KEY (user_id) REFERENCES users(user_id)
);
""")

users_data = [(1, "rahul_sharma"), (2, "priya_patel"), (3, "amit_verma"), (4, "sneha_reddy"), (5, "vikram_m")]
orders_data = [
    (101, 1, 450.00),
    (102, 1, 750.00),
    (103, 2, 1200.00),
    (104, 2, 800.00),
    (105, 3, 350.00),
    (106, 4, 950.00),
    (107, 4, 1100.00),
    (108, 4, 600.00)
]
cursor.executemany("INSERT INTO users (user_id, username) VALUES (?, ?);", users_data)
cursor.executemany("INSERT INTO orders (order_id, user_id, amount) VALUES (?, ?, ?);", orders_data)
conn.commit()

# 2. Comprehensive LEFT JOIN query
query_task2 = """
SELECT 
    u.username,
    COALESCE(SUM(o.amount), 0.00) AS total_order_amount
FROM users u
LEFT JOIN orders o ON u.user_id = o.user_id
GROUP BY u.user_id, u.username
ORDER BY total_order_amount DESC;
"""
df_task2 = pd.read_sql_query(query_task2, conn)
print("--- Total Order Amount per User (LEFT JOIN) ---")
display(df_task2)


--- Total Order Amount per User (LEFT JOIN) ---


,username,total_order_amount
0,sneha_reddy,2650.0
1,priya_patel,2000.0
2,rahul_sharma,1200.0
3,amit_verma,350.0
4,vikram_m,0.0


## Task 3: Above-Average Rated Restaurants (Scalar Subquery)
> **Task 3:** Write a SQL subquery to find the names of all restaurants from a `restaurants` table (`id`, `name`, `rating`) whose rating is higher than the average rating of all restaurants.

### Interview Key Concept: Uncorrelated Scalar Subquery
- The inner query `(SELECT AVG(rating) FROM restaurants)` executes **once** and evaluates to a single numeric scalar value (e.g., $4.17$).
- The outer query scans each row of `restaurants` and compares `rating > 4.17`.

In [3]:
# 1. Create restaurants table
cursor.execute("DROP TABLE IF EXISTS restaurants;")
cursor.execute("""
CREATE TABLE restaurants (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    rating DECIMAL(3, 2) NOT NULL
);
""")

restaurants_data = [
    (1, "Gulati", 4.80),
    (2, "Punjab Grill", 4.50),
    (3, "Toscano", 4.60),
    (4, "Haldirams", 3.80),
    (5, "Subway", 3.70),
    (6, "Chai Point", 3.60)
]
cursor.executemany("INSERT INTO restaurants (id, name, rating) VALUES (?, ?, ?);", restaurants_data)
conn.commit()

# 2. Query with scalar subquery
query_task3 = """
SELECT 
    name,
    rating
FROM restaurants
WHERE rating > (
    SELECT AVG(rating) 
    FROM restaurants
)
ORDER BY rating DESC;
"""
df_task3 = pd.read_sql_query(query_task3, conn)

avg_rating = pd.read_sql_query("SELECT AVG(rating) AS avg_r FROM restaurants;", conn)["avg_r"].iloc[0]
print(f"Overall Average Rating: {round(avg_rating, 2)}")
print("--- Restaurants with Rating Above Average ---")
display(df_task3)


Overall Average Rating: 4.17
--- Restaurants with Rating Above Average ---


,name,rating
0,Gulati,4.8
1,Toscano,4.6
2,Punjab Grill,4.5


## Task 4: User Running Total (Window Function)
> **Task 4:** Using a `transactions` table (`id`, `user_id`, `amount`, `transaction_date`), write a SQL query with a window function to display each user's transaction amount and their running total (cumulative sum) ordered by `transaction_date`.

### Interview Key Concept: Frame Specification
- `PARTITION BY user_id`: Partitions the window calculation so running totals reset for each user.
- `ORDER BY transaction_date ASC`: Dictates chronological order of accumulation.
- `ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW`: Explicitly defines the rolling accumulation window.

In [4]:
# 1. Create transactions table
cursor.execute("DROP TABLE IF EXISTS transactions;")
cursor.execute("""
CREATE TABLE transactions (
    id INTEGER PRIMARY KEY,
    user_id INTEGER NOT NULL,
    amount DECIMAL(10, 2) NOT NULL,
    transaction_date DATE NOT NULL
);
""")

transactions_data = [
    (1, 101, 500.00,  "2026-03-01"),
    (2, 101, 300.00,  "2026-03-05"),
    (3, 101, 700.00,  "2026-03-10"),
    (4, 102, 1000.00, "2026-03-02"),
    (5, 102, 450.00,  "2026-03-08"),
    (6, 102, 550.00,  "2026-03-15")
]
cursor.executemany("INSERT INTO transactions (id, user_id, amount, transaction_date) VALUES (?, ?, ?, ?);", transactions_data)
conn.commit()

# 2. Window cumulative sum query
query_task4 = """
SELECT 
    user_id,
    id AS transaction_id,
    transaction_date,
    amount,
    SUM(amount) OVER (
        PARTITION BY user_id 
        ORDER BY transaction_date ASC
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS running_total
FROM transactions
ORDER BY user_id ASC, transaction_date ASC;
"""
df_task4 = pd.read_sql_query(query_task4, conn)
print("--- User Transactions with Running Total ---")
display(df_task4)


--- User Transactions with Running Total ---


,user_id,transaction_id,transaction_date,amount,running_total
0,101,1,2026-03-01,500,500
1,101,2,2026-03-05,300,800
2,101,3,2026-03-10,700,1500
3,102,4,2026-03-02,1000,1000
4,102,5,2026-03-08,450,1450
5,102,6,2026-03-15,550,2000


## Task 5: Flipkart Product Query Optimization
> **Task 5:** List two optimizations you would apply to speed up a query that filters Flipkart products by category and price, and briefly explain how each helps.  
> *Hint: Think about indexes and query structure.*

### Scenario:
Target query:
```sql
SELECT product_id, title, price, category 
FROM products 
WHERE category = 'Electronics' AND price BETWEEN 1000 AND 5000;
```

---

### Optimization 1: Composite B-Tree Index on `(category, price)`
- **Design & Ordering Rule:** In composite indexes, the **Equality column (`category = ...`) must precede the Range column (`price BETWEEN ...`)**:
  ```sql
  CREATE INDEX idx_products_category_price ON products (category, price);
  ```
- **How it helps:**
  1. The database engine performs an **Index Seek** directly to the `'Electronics'` node in logarithmic time $\mathcal{O}(\log N)$.
  2. Because entries within `'Electronics'` are pre-sorted by `price`, the engine performs a bounded leaf-node scan strictly between $1,000$ and $5,000$.
  3. It avoids scanning millions of rows in other categories, reducing I/O and latency from seconds to milliseconds.

---

### Optimization 2: Covering Index & SARGable Query Structure
- **Design:** Ensure the query predicates are **SARGable** (Search Argument Able):
  - Never wrap indexed columns in functions (e.g., avoid `WHERE LOWER(category) = 'electronics'` or `WHERE CAST(price AS INT) ...`), which blinds the query optimizer and forces a full table scan.
  - In engines supporting covering indexes (PostgreSQL, MySQL, SQL Server), include retrieved columns:
    ```sql
    CREATE INDEX idx_covering ON products (category, price) INCLUDE (title, product_id);
    ```
- **How it helps:**
  - Facilitates an **Index-Only Scan**. The query engine fulfills the entire query directly from index memory pages without performing random I/O heap fetches to the underlying table pages.

In [5]:
# 1. Create products table and populate sample inventory
cursor.execute("DROP TABLE IF EXISTS products;")
cursor.execute("""
CREATE TABLE products (
    product_id INTEGER PRIMARY KEY,
    title TEXT NOT NULL,
    category TEXT NOT NULL,
    price DECIMAL(10, 2) NOT NULL,
    stock INTEGER NOT NULL
);
""")

sample_products = [
    (1, "Wireless Earbuds", "Electronics", 1499.00, 50),
    (2, "Smart Watch", "Electronics", 2999.00, 30),
    (3, "Bluetooth Speaker", "Electronics", 1899.00, 40),
    (4, "Gaming Laptop", "Electronics", 65000.00, 10),
    (5, "Running Shoes", "Footwear", 2499.00, 100),
    (6, "Cotton T-Shirt", "Apparel", 699.00, 200)
]
cursor.executemany("INSERT INTO products (product_id, title, category, price, stock) VALUES (?, ?, ?, ?, ?);", sample_products)
conn.commit()

# 2. Implement Optimization 1: Composite B-Tree Index
cursor.execute("CREATE INDEX idx_products_category_price ON products (category, price);")
conn.commit()

# 3. Verify Execution Plan using EXPLAIN QUERY PLAN
explain_df = pd.read_sql_query("""
EXPLAIN QUERY PLAN
SELECT product_id, title, price, category 
FROM products 
WHERE category = 'Electronics' AND price BETWEEN 1000 AND 5000;
""", conn)

print("--- Query Execution Plan Verification ---")
display(explain_df)

# Close database connection
conn.close()
print("\n[OK] Database connection closed cleanly.")


--- Query Execution Plan Verification ---


,id,parent,notused,detail
0,3,0,49,SEARCH products USING INDEX idx_products_categ...



[OK] Database connection closed cleanly.


---

## 🎯 SQL Interview Cheat Sheet: Order of Query Execution

In SQL technical interviews, understanding the difference between **Lexical Order** (how SQL is written) and **Logical Execution Order** (how the DBMS engine actually processes the query) is essential:

```text
1. FROM               -> Identify base tables and table sources
2. ON                 -> Apply join conditions
3. JOIN               -> Merge relational datasets
4. WHERE              -> Filter raw table rows before grouping
5. GROUP BY           -> Partition dataset into categorical groups
6. HAVING             -> Filter groups based on aggregate results
7. SELECT             -> Evaluate expressions and project output columns
8. WINDOW (OVER)      -> Compute window calculations and rankings
9. DISTINCT           -> Eliminate duplicate rows
10. ORDER BY          -> Sort final result set
11. LIMIT / OFFSET    -> Truncate to top N output rows
```
